# ⚡ Northside — a nanosecond tabular classifier (Rapid Lightning family)

**Northside** is the head-aware member of the *Rapid Lightning* family: a sparsely-gated mixture of two experts —
a genetic-programming model whose members are evolved as *signal generators* for a linear head, and an XGBoost model —
joined by a competence gate. It compiles to a **~15 KB, zero-dependency C `.so`** that predicts in **tens of nanoseconds**.

- **RLN** = Northside as the full gated hybrid (RL + XGBoost).
- **RLNF** = the same with XGBoost compiled via *XGForge* into the same nanosecond envelope.

**What this notebook is for:** grab any section you need. Engine · hybrid · the two `.so` compilers · a "will it help my data?" advisory ·
a copy-paste "use on your own data" path · a rigorous CV+significance harness · and a speed shootout against **XGBoost, LightGBM,
treelite/TL2cgen, and Timber**. Honest headline: **Northside's speed/size/energy win is unconditional; its accuracy edge over XGBoost
is conditional on headroom** (see §8). Licensed under AGPL-3.0 — see §9. Run top-to-bottom, or cherry-pick.

### 0 · Install

In [ ]:
!pip install -q xgboost lightgbm treelite tl2cgen

### 1 · The Rapid Lightning engine — head-aware genetic programming
Members are evolved to maximize `corr(signal ⟂ pool, residual)²` — i.e. functional gradient boosting in feature space. Individually they're near-chance; together they feed a linear head.

In [ ]:
#!/usr/bin/env python3
"""
Substrate experiment: does an `ifte` value-routing primitive help, and
does scaling members ("rainforest") keep paying off?

Self-contained (no gp_lab import) so the op alphabet is configurable.
Numpy + sklearn only — runs anywhere, CPU is fine.
"""
import random, time
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.datasets import make_classification, fetch_openml
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

# ---------------- configurable op alphabet ----------------
BASE = {'fma':3,'add':2,'mul':2,'sub':2,'max':2,'min':2,'neg':1,'abs':1,'clamp':1,'lt':2}
OPS = dict(BASE); OP_KEYS = list(BASE)
def set_ops(use_ifte):
    global OPS, OP_KEYS
    OPS = dict(BASE)
    if use_ifte: OPS['ifte'] = 3          # value routing: cond>0 ? a : b
    OP_KEYS = list(OPS)

class Node:
    __slots__=['t','v','c']
    def __init__(s,t,v,c=None): s.t=t; s.v=v; s.c=c or []
    def ev(s,X):
        if s.t=='in': return X[:, s.v % X.shape[1]]
        if s.t=='cn': return np.full(X.shape[0], s.v, dtype=X.dtype)
        a=[ch.ev(X) for ch in s.c]; op=s.v
        if   op=='fma':  r=a[0]*a[1]+a[2]
        elif op=='add':  r=a[0]+a[1]
        elif op=='mul':  r=a[0]*a[1]
        elif op=='sub':  r=a[0]-a[1]
        elif op=='max':  r=np.maximum(a[0],a[1])
        elif op=='min':  r=np.minimum(a[0],a[1])
        elif op=='neg':  r=-a[0]
        elif op=='abs':  r=np.abs(a[0])
        elif op=='clamp':r=np.clip(a[0],0.,1.)
        elif op=='lt':   r=(a[0]<a[1]).astype(X.dtype)
        elif op=='ifte': r=np.where(a[0]>0.0, a[1], a[2])     # the new primitive
        else:            r=a[0]
        return np.clip(r,-1e6,1e6)
    def size(s): return 1+sum(ch.size() for ch in s.c)
    def depth(s): return 1 if not s.c else 1+max(ch.depth() for ch in s.c)
    def clone(s): return Node(s.t,s.v,[ch.clone() for ch in s.c])
    def nodes(s):
        o=[s]
        for ch in s.c: o.extend(ch.nodes())
        return o

class Ind:
    __slots__=['trees','fitness','train_acc']
    def __init__(s,trees): s.trees=trees; s.fitness=0.0; s.train_acc=0.0
    def scores(s,X): return np.stack([t.ev(X) for t in s.trees],axis=1)
    def classify(s,X): return s.scores(X).argmax(1)
    def total(s): return sum(t.size() for t in s.trees)
    def clone(s):
        n=Ind([t.clone() for t in s.trees]); n.fitness=s.fitness; n.train_acc=s.train_acc; return n

def rtree(md,ins,d=0):
    if d>=md or (d>1 and random.random()<0.3):
        return Node('in',random.randint(0,ins-1)) if random.random()<0.7 else Node('cn',random.uniform(-2,2))
    op=random.choice(OP_KEYS); return Node('op',op,[rtree(md,ins,d+1) for _ in range(OPS[op])])
def rind(nc,md,ins): return Ind([rtree(md,ins) for _ in range(nc)])
def mut(ind,md,ins):
    m=ind.clone(); ti=random.randint(0,len(m.trees)-1); ns=m.trees[ti].nodes()
    if len(ns)<=1 or random.random()<0.3: m.trees[ti]=rtree(md,ins)
    else:
        t=random.choice(ns)
        if t.t=='cn': t.v+=random.gauss(0,.5)
        elif t.t=='in': t.v=random.randint(0,ins-1)
        else:
            n=rtree(min(3,md-1),ins); t.t,t.v,t.c=n.t,n.v,n.c
    return m
def xo(p1,p2,md,ins):
    ch=p1.clone(); ti=random.randint(0,len(ch.trees)-1); n1,n2=ch.trees[ti].nodes(),p2.trees[ti].nodes()
    if len(n1)>1 and len(n2)>1:
        tg=random.choice(n1); dn=random.choice(n2).clone(); tg.t,tg.v,tg.c=dn.t,dn.v,dn.c
    if ch.trees[ti].depth()>md+2: ch.trees[ti]=rtree(md,ins)
    return ch
def tsel(pop,sz): 
    b=None
    for _ in range(sz):
        c=random.choice(pop)
        if b is None or c.fitness>b.fitness: b=c
    return b

# ---- head-aware fitness (production) ----
def stack(ms,X): return np.concatenate([m.scores(X) for m in ms],axis=1).astype(np.float64)
def pool(ms,X,y,nc):
    N=X.shape[0]; Y1=np.eye(nc)[y]; cols=[np.ones((N,1))]
    if ms:
        Ph=StandardScaler().fit_transform(stack(ms,X)); cols.append(Ph)
        lr=LogisticRegression(max_iter=2000,C=1.,solver='lbfgs').fit(Ph,y); P=np.zeros((N,nc))
        for j,c in enumerate(lr.classes_): P[:,c]=lr.predict_proba(Ph)[:,j]
    else: P=np.tile(np.bincount(y,minlength=nc)/N,(N,1))
    B=np.concatenate(cols,1); Bp=B@np.linalg.inv(B.T@B+1e-3*np.eye(B.shape[1])); return (Y1-P),Bp,B
def headaware(R,Bp,B,pars):
    Rc=R-R.mean(0); Rn=np.sqrt((Rc**2).sum(0))+1e-12
    def ev(pop,X):
        for ind in pop:
            try:
                S=np.nan_to_num(ind.scores(X).astype(np.float64),nan=0.,posinf=1e6,neginf=-1e6)
                Sp=S-Bp@(B.T@S); Sc=Sp-Sp.mean(0); Sn=np.sqrt((Sc**2).sum(0))+1e-12
                corr=(Sc*Rc).sum(0)/(Sn*Rn); ind.fitness=float((corr**2).sum())-ind.total()*pars; ind.train_acc=ind.fitness
            except Exception: ind.fitness=-1e9
    return ev

def evolve_member(X,y,cfg,ev_factory,mnum):
    ins=X.shape[1]; nc=int(y.max())+1; random.seed(cfg['seed']+mnum*1000)
    ppi=cfg['pop']//cfg['islands']
    isl=[[rind(nc,cfg['depth'],ins) for _ in range(ppi)] for _ in range(cfg['islands'])]
    best=None; bf=-1e18; ev=ev_factory
    for gen in range(1,cfg['gens']+1):
        for i in range(len(isl)):
            pop=isl[i]; ev(pop,X); pop.sort(key=lambda z:z.fitness,reverse=True)
            if pop[0].fitness>bf: bf=pop[0].fitness; best=pop[0].clone()
            new=[]; ec=max(2,int(len(pop)*0.05))
            for j in range(ec): new.append(pop[j].clone())
            while len(new)<len(pop):
                p1,p2=tsel(pop,cfg['tour']),tsel(pop,cfg['tour'])
                ch=xo(p1,p2,cfg['depth'],ins) if random.random()<0.85 else p1.clone()
                if random.random()<0.20: ch=mut(ch,cfg['depth'],ins)
                new.append(ch)
            isl[i]=new
        if gen%8==0:
            ni=len(isl)
            for i in range(ni):
                s=sorted(isl[i],key=lambda z:z.fitness,reverse=True); t=sorted(isl[(i+1)%ni],key=lambda z:z.fitness)
                for m in range(min(3,len(s),len(t))): t[m]=s[m].clone()
                isl[(i+1)%ni]=t
    return best

def seq_headaware(X,y,cfg):
    nc=int(y.max())+1; ms=[]
    for m in range(1,cfg['members']+1):
        R,Bp,B=pool(ms,X,y,nc); ms.append(evolve_member(X,y,cfg,headaware(R,Bp,B,cfg['pars']),m))
    return ms

def lr_head(ms,Xtr,ytr,Xte,yte):
    Ztr,Zte=stack(ms,Xtr),stack(ms,Xte); sc=StandardScaler().fit(Ztr)
    lr=LogisticRegression(max_iter=2000,C=1.,solver='lbfgs').fit(sc.transform(Ztr),ytr)
    p=lr.predict_proba(sc.transform(Zte))[:,1]
    return float(lr.score(sc.transform(Zte),yte)), float(roc_auc_score(yte,p))

def load_synth(seed):
    X,y=make_classification(n_samples=9000,n_features=20,n_informative=10,n_redundant=4,
                            n_clusters_per_class=3,class_sep=0.9,flip_y=0.02,random_state=seed)
    Xtr,Xte,ytr,yte=train_test_split(X.astype(np.float32),y,test_size=0.3,random_state=seed,stratify=y)
    return Xtr,ytr.astype(int),Xte,yte.astype(int)

_ADULT=None
def load_adult(seed,sub=6000):
    global _ADULT
    if _ADULT is None:
        a=fetch_openml('adult',version=2,as_frame=True,parser='auto'); X=a.data.copy()
        y=(a.target.astype(str).str.replace('.','',regex=False)=='>50K').astype(int).values
        X=X.replace('?',np.nan); cat=X.select_dtypes(include=['object','category']).columns.tolist()
        num=X.select_dtypes(include=['number']).columns.tolist()
        for c in cat: X[c]=X[c].fillna(X[c].mode().iloc[0])
        for c in num: X[c]=X[c].fillna(X[c].median())
        _ADULT=(X,y,cat,num)
    X,y,cat,num=_ADULT
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
    pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cat)])
    pre.fit(Xtr); Xtr=pre.transform(Xtr).astype(np.float32); Xte=pre.transform(Xte).astype(np.float32)
    Xtr,_,ytr,_=train_test_split(Xtr,ytr,train_size=sub,random_state=seed,stratify=ytr)
    return Xtr,ytr.astype(int),Xte,yte.astype(int)

def base_cfg(seed,**kw):
    c=dict(seed=seed,members=8,pop=130,gens=15,depth=5,islands=2,tour=5,pars=0.0001); c.update(kw); return c


### 2 · Northside — the gated RL + XGBoost hybrid (RLN / RLNF)
Two experts, a gate trained to predict which one wins per-sample, and several serving modes (`proba_soft` = the accurate blend).

In [ ]:
#!/usr/bin/env python3
"""RoutedHybrid — sparse two-expert MoE: Rapid Lightning (fast) + XGBoost (heavy),
a cheap gate routing per-sample. Imports the engine from gp_substrate for local
testing; the Colab notebook inlines that engine instead."""
import time
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml
from xgboost import XGBClassifier

# engine (inlined in the notebook; imported here)
set_ops(False)
HAVE_XGFORGE = 'xgforge' in globals()

def _logloss(p, y):
    p = np.clip(p, 1e-6, 1-1e-6)
    return -(y*np.log(p) + (1-y)*np.log(1-p))

class RoutedHybrid:
    """Fit RL + XGB experts and a gate; serve via sparse hard routing or soft blend.

    gate='lr'   -> cheap linear gate (dot product; ~free, the latency-safe choice)
    gate='tree' -> small GBM gate (better routing quality, heavier)
    """
    def __init__(self, rl_kw=None, xgb_kw=None, gate='lr', seed=0):
        self.rl_kw = rl_kw or dict(members=10, pop=120, gens=14)
        self.xgb_kw = xgb_kw or dict(n_estimators=400, max_depth=6, learning_rate=0.1,
                                     tree_method='hist', eval_metric='logloss',
                                     n_jobs=-1, verbosity=0)
        self.gate_kind = gate
        self.seed = seed

    def fit(self, Xtr, ytr, val_frac=0.2):
        Xb, Xv, yb, yv = train_test_split(Xtr, ytr, test_size=val_frac,
                                          stratify=ytr, random_state=self.seed)
        # --- expert 1: Rapid Lightning (head-aware GP -> LR head) ---
        self.members = seq_headaware(Xb, yb, base_cfg(self.seed, **self.rl_kw))
        self.rl_sc = StandardScaler().fit(stack(self.members, Xb))
        self.rl_lr = LogisticRegression(max_iter=2000, C=1., solver='lbfgs').fit(
            self.rl_sc.transform(stack(self.members, Xb)), yb)
        # --- expert 2: XGBoost ---
        self.xgb = XGBClassifier(random_state=self.seed, **self.xgb_kw).fit(Xb, yb)
        # --- gate: predict from x which expert has lower loss (trained on val) ---
        pRLv, pXGv = self._rl(Xv), self.xgb.predict_proba(Xv)[:, 1]
        tgt = (_logloss(pRLv, yv) < _logloss(pXGv, yv)).astype(int)   # 1 = RL better
        if self.gate_kind == 'lr':
            self.gate_sc = StandardScaler().fit(Xv)
            self.gate = LogisticRegression(max_iter=1000, C=1.).fit(self.gate_sc.transform(Xv), tgt)
        else:
            self.gate = XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.2,
                                      tree_method='hist', n_jobs=-1, verbosity=0,
                                      random_state=self.seed).fit(Xv, tgt)
        # --- global stacker (rating baseline) ---
        self.stacker = LogisticRegression(max_iter=1000).fit(np.c_[pRLv, pXGv], yv)
        return self

    def _rl(self, X):  # RL probability of class 1
        return self.rl_lr.predict_proba(self.rl_sc.transform(stack(self.members, X)))[:, 1]
    def gate_score(self, X):  # P(RL is the better expert | x)
        Z = self.gate_sc.transform(X) if self.gate_kind == 'lr' else X
        return self.gate.predict_proba(Z)[:, 1]

    # ---- serving modes ----
    def proba_xgb(self, X):  return self.xgb.predict_proba(X)[:, 1]
    def proba_soft(self, X):
        g = self.gate_score(X); return g*self._rl(X) + (1-g)*self.proba_xgb(X)
    def proba_stack(self, X):
        return self.stacker.predict_proba(np.c_[self._rl(X), self.proba_xgb(X)])[:, 1]
    def compile_xgforge(self):
        """RLNF: compile the trained XGBoost to a fp32 .so (rank-faithful, ~150ns).
        Same model -> same AUC/accuracy; only the inference engine changes."""
        if not HAVE_XGFORGE:
            raise RuntimeError('xgforge unavailable (needs the xgforge pipeline + gcc)')
        self.xgf = xgforge(self.xgb)
        return self
    def _heavy(self, X, fast):
        if fast and getattr(self, 'xgf', None) is not None:
            return self.xgf.predict_proba(np.ascontiguousarray(X, dtype=np.float32))
        return self.xgb.predict_proba(X)[:, 1]
    def route_hard(self, X, skip_frac, fast=False):
        """Route the top `skip_frac` of samples (by gate score) to RL-ONLY; the heavy
        expert is only invoked on the rest. fast=True uses the XGForge .so (RLNF).
        Returns (proba, to_rl_mask)."""
        g = self.gate_score(X); tau = np.quantile(g, 1-skip_frac); to_rl = g >= tau
        proba = np.empty(len(X), dtype=float); pRL = self._rl(X)
        proba[to_rl] = pRL[to_rl]
        if (~to_rl).any():
            proba[~to_rl] = self._heavy(X[~to_rl], fast)   # sparse: heavy skipped on to_rl
        return proba, to_rl

# ---------------- data loaders ----------------
def _prep(Xdf, y, seed, sub=None):
    import pandas as pd
    cat = Xdf.select_dtypes(include=['object', 'category']).columns.tolist()
    num = Xdf.select_dtypes(include=['number']).columns.tolist()
    for c in cat: Xdf[c] = Xdf[c].fillna(Xdf[c].mode().iloc[0])
    for c in num: Xdf[c] = Xdf[c].fillna(Xdf[c].median())
    Xtr_df, Xte_df, ytr, yte = train_test_split(Xdf, y, test_size=0.2, random_state=42, stratify=y)
    pre = ColumnTransformer([('n', StandardScaler(), num),
                             ('c', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat)])
    pre.fit(Xtr_df)
    Xtr = pre.transform(Xtr_df).astype('float32'); Xte = pre.transform(Xte_df).astype('float32')
    if sub and sub < len(Xtr):
        Xtr, _, ytr, _ = train_test_split(Xtr, ytr, train_size=sub, random_state=seed, stratify=ytr)
    return Xtr, ytr.astype(int), Xte, yte.astype(int)

def load_bank(seed, sub=6000):
    b = fetch_openml('bank-marketing', version=1, as_frame=True, parser='auto')
    y = (b.target.astype(str).isin(['2', 'yes'])).astype(int).values
    return _prep(b.data.copy(), y, seed, sub)

def load_openml(data_id, seed=0, sub=None):
    """Verified OpenML loader by stable data_id. Binary targets -> 0/1."""
    d = fetch_openml(data_id=data_id, as_frame=True, parser='auto')
    y = d.target.astype('category').cat.codes.values
    return _prep(d.data.copy(), y, seed, sub)

def load_breast(seed=0, sub=None):
    from sklearn.datasets import load_breast_cancer
    import pandas as pd
    bc = load_breast_cancer(); X = pd.DataFrame(bc.data, columns=bc.feature_names)
    return _prep(X, bc.target.astype(int), seed, sub)

# Verified well-established binary sets (data_ids checked live). seed varies model
# (and subsample on the big two); small sets share a fixed train/test split.
DATASETS = {
    'pima-diabetes':  lambda s: load_openml(37,  s),            # 768 x 8
    'german-credit':  lambda s: load_openml(31,  s),            # 1000 x 20
    'breast-cancer':  lambda s: load_breast(s),                 # 569 x 30 (sklearn builtin)
    'spambase':       lambda s: load_openml(44,  s),            # 4601 x 57
    'adult':          lambda s: load_openml(1590, s, sub=6000), # 48842 -> 6000
    'bank-marketing': lambda s: load_openml(1461, s, sub=6000), # 45211 -> 6000
}

def load_csv(path, target, seed, sub=None):
    import pandas as pd
    df = pd.read_csv(path); ys = df[target]; X = df.drop(columns=[target])
    y = ys.astype('category').cat.codes.values if ys.dtype == object else ys.values
    assert len(set(y)) == 2, "this reference is binary; map your target to two classes first"
    return _prep(X, y.astype(int), seed, sub)

# ---------------- evaluation ----------------
def evaluate(loader, seeds=(1, 2, 3), skip_fracs=(0.1, 0.2, 0.3), gate='lr', rl_kw=None, label=""):
    rows = {'xgb_acc': [], 'xgb_auc': [], 'stack_acc': [], 'stack_auc': [],
            'soft_acc': [], 'soft_auc': [], 'gate_auc': []}
    fr = {f: {'acc': [], 'auc': []} for f in skip_fracs}
    for s in seeds:
        Xtr, ytr, Xte, yte = loader(s)
        h = RoutedHybrid(gate=gate, rl_kw=rl_kw, seed=s).fit(Xtr, ytr)
        pXG = h.proba_xgb(Xte); pST = h.proba_stack(Xte); pSO = h.proba_soft(Xte)
        rows['xgb_acc'].append(accuracy_score(yte, (pXG > .5).astype(int)))
        rows['xgb_auc'].append(roc_auc_score(yte, pXG))
        rows['stack_acc'].append(accuracy_score(yte, (pST > .5).astype(int)))
        rows['stack_auc'].append(roc_auc_score(yte, pST))
        rows['soft_acc'].append(accuracy_score(yte, (pSO > .5).astype(int)))
        rows['soft_auc'].append(roc_auc_score(yte, pSO))
        # gate diagnostic
        g = h.gate_score(Xte); tgt = (_logloss(h._rl(Xte), yte) < _logloss(pXG, yte)).astype(int)
        rows['gate_auc'].append(roc_auc_score(tgt, g) if len(set(tgt)) > 1 else .5)
        for f in skip_fracs:
            p, _ = h.route_hard(Xte, f)
            fr[f]['acc'].append(accuracy_score(yte, (p > .5).astype(int)))
            fr[f]['auc'].append(roc_auc_score(yte, p))
    M = lambda k: float(np.mean(rows[k])); S = lambda k: float(np.std(rows[k]))
    print(f"\n=== {label or loader.__name__} | {len(seeds)} seeds | gate='{gate}' ===")
    print(f"  XGB-alone:    acc {M('xgb_acc'):.4f}±{S('xgb_acc'):.4f}   AUC {M('xgb_auc'):.4f}")
    print(f"  global stack: acc {M('stack_acc'):.4f}            AUC {M('stack_auc'):.4f}")
    print(f"  soft MoE:     acc {M('soft_acc'):.4f}            AUC {M('soft_auc'):.4f}")
    print(f"  gate_auc (can x pick the winner?): {M('gate_auc'):.4f}  (0.5=hopeless)")
    print(f"  --- hard routing frontier (route top-g to RL-only, skip XGB) ---")
    print(f"  {'XGB-skip':>9s} {'route_acc':>10s} {'route_auc':>10s} {'Δacc vs XGB':>12s}")
    out = {'skip': list(skip_fracs), 'route_acc': [], 'route_auc': [],
           'xgb_acc': M('xgb_acc'), 'xgb_auc': M('xgb_auc'), 'gate_auc': M('gate_auc'),
           'stack_auc': M('stack_auc'), 'soft_auc': M('soft_auc')}
    for f in skip_fracs:
        ra, ru = float(np.mean(fr[f]['acc'])), float(np.mean(fr[f]['auc']))
        out['route_acc'].append(ra); out['route_auc'].append(ru)
        tag = "  faster+better" if ra >= M('xgb_acc') else ""
        print(f"  {f*100:>8.0f}% {ra:>10.4f} {ru:>10.4f} {ra-M('xgb_acc'):>+12.4f}{tag}")
    return out

def sweep(datasets=None, seeds=(1, 2), at_skip=0.3, gate='lr', rl_kw=None):
    """Run RLN across a battery of datasets; report a per-dataset verdict."""
    datasets = datasets or DATASETS
    summary = []
    for name, loader in datasets.items():
        r = evaluate(loader, seeds=seeds, skip_fracs=(0.1, 0.2, 0.3), gate=gate, rl_kw=rl_kw, label=name)
        best = max(r['stack_auc'], r['soft_auc'], max(r['route_auc']))  # best hybrid mode
        i = r['skip'].index(at_skip) if at_skip in r['skip'] else len(r['skip'])-1
        summary.append((name, r['xgb_auc'], best, r['route_auc'][i], r['gate_auc']))
    print('\n' + '=' * 78)
    print('SWEEP SUMMARY — RLN (Rapid Lightning + gated XGBoost) vs XGBoost-alone')
    print('=' * 78)
    print(f"  {'dataset':16s} {'XGB AUC':>9s} {'RLN best':>9s} {'delta':>8s} {'gate_auc':>9s}  verdict")
    for name, xa, best, ra30, ga in summary:
        d = best - xa
        v = 'RLN wins' if d > 5e-4 else ('~tie' if abs(d) <= 5e-4 else 'XGB wins')
        print(f"  {name:16s} {xa:>9.4f} {best:>9.4f} {d:>+8.4f} {ga:>9.4f}  {v}")
    print("  (RLN best = best of stacker / soft-MoE / hard-route frontier. gate_auc>~0.7")
    print("   means hard-routing for latency also pays; lower means lean on the stacker.)")
    return summary

def measure_latency(hybrid, X, t_rl_us=0.3, n=2000, warm=200):
    """Single-row latencies (this env) + cost model. Absolute µs are env/overhead
    dependent; the durable fact is the gate:XGB ratio (a dot product << 400 trees)."""
    rng = np.random.default_rng(0); rows = X[rng.integers(0, len(X), n)].astype(np.float32)
    def bench(fn):
        for i in range(warm): fn(rows[i % len(rows)])
        t = time.perf_counter()
        for i in range(n): fn(rows[i % len(rows)])
        return (time.perf_counter()-t)/n*1e6
    t_xgb = bench(lambda r: hybrid.xgb.get_booster().inplace_predict(r.reshape(1, -1)))
    if hybrid.gate_kind == 'lr':
        W, b = hybrid.gate.coef_[0], hybrid.gate.intercept_[0]
        mu, sd = hybrid.gate_sc.mean_, hybrid.gate_sc.scale_
        t_gate = bench(lambda r: 1/(1+np.exp(-(W @ ((r-mu)/sd) + b))))
    else:
        t_gate = bench(lambda r: hybrid.gate.get_booster().inplace_predict(r.reshape(1, -1)))
    return {'t_xgb': t_xgb, 't_gate': t_gate, 't_rl': t_rl_us, 'ratio': t_gate/t_xgb}


### 3 · Compile to a nanosecond `.so`
`xgforge` compiles a trained XGBoost to fp32 C (rank-faithful). `rl_to_c` compiles Northside's RL core (GP trees → C, scaler+LR folded in). `clamp_ops=False` is the fast inference default — the per-op clamp only matters during evolution.

In [ ]:
"""XGForge pipeline (from XGForge.ipynb) — XGBClassifier -> compiled fp32 .so."""
import numpy as np, os, tempfile, ctypes, subprocess

def fmt_f(v):
    v=float(v); s=f"{v:.9g}"
    if '.' not in s and 'e' not in s and 'E' not in s and 'n' not in s and 'i' not in s: s=f"{v:.1f}"
    return s+'f'

def compile_c(src, prefix='xgforge_'):
    d=tempfile.mkdtemp(prefix=prefix); sp,op=os.path.join(d,'p.c'),os.path.join(d,'p.so')
    open(sp,'w').write(src)
    subprocess.run(['gcc','-O3','-march=native','-ffast-math','-shared','-fPIC','-o',op,sp],check=True)
    return op

class ForgeAOT:
    def __init__(self, so_path):
        self.so_path=so_path; self.lib=ctypes.CDLL(so_path)
        self.lib.predict_proba_one.restype=ctypes.c_float
        self.lib.predict_proba_one.argtypes=[ctypes.POINTER(ctypes.c_float)]
        self.lib.predict_proba_time_ns.restype=ctypes.c_double
        self.lib.predict_proba_time_ns.argtypes=[ctypes.POINTER(ctypes.c_float),ctypes.c_int32,ctypes.c_int32]
    def predict_proba_one(self,x):
        x32=np.ascontiguousarray(x,dtype=np.float32)
        return float(self.lib.predict_proba_one(x32.ctypes.data_as(ctypes.POINTER(ctypes.c_float))))
    def predict_proba(self,X):
        return np.array([self.predict_proba_one(X[i]) for i in range(len(X))])
    def time_ns(self,x,nw=5000,ni=200000):
        x32=np.ascontiguousarray(x,dtype=np.float32)
        return float(self.lib.predict_proba_time_ns(x32.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),nw,ni))
    @property
    def so_kb(self): return os.path.getsize(self.so_path)/1024

def _timing_func_c():
    return '''
double predict_proba_time_ns(const float* X, int32_t nw, int32_t ni) {
    volatile float sink = 0;
    for (int32_t i = 0; i < nw; ++i) sink += predict_proba_one(X);
    struct timespec t0, t1;
    clock_gettime(CLOCK_MONOTONIC, &t0);
    for (int32_t i = 0; i < ni; ++i) sink += predict_proba_one(X);
    clock_gettime(CLOCK_MONOTONIC, &t1);
    (void)sink;
    double dt = (double)(t1.tv_sec-t0.tv_sec)*1e9 + (double)(t1.tv_nsec-t0.tv_nsec);
    return dt / (double)ni;
}'''

def xgforge(xgb_model):
    booster=xgb_model.get_booster(); bs=booster.attr('base_score')
    base_score=float(bs) if bs else 0.5
    base_logit=float(np.log(base_score/(1-base_score))) if 0<base_score<1 else base_score
    df=booster.trees_to_dataframe(); tree_ids=sorted(df['Tree'].unique())
    lines=['#include <math.h>','#include <stdint.h>','#include <time.h>','']
    for tid in tree_ids:
        tdf=df[df['Tree']==tid]; nodes={int(r['Node']):r for _,r in tdf.iterrows()}; root=int(tdf['Node'].min())
        lines.append(f'static inline float tree_{tid}(const float* X) {{')
        def recurse(nid,indent):
            row=nodes[nid]; feat=row['Feature']
            if feat=='Leaf': lines.append('    '*indent+f'return {fmt_f(row["Gain"])};'); return
            try: fi=int(str(feat).lstrip('f'))
            except ValueError: fi=0
            yn=int(str(row['Yes']).split('-')[1]); nn=int(str(row['No']).split('-')[1])
            lines.append('    '*indent+f'if (X[{fi}] < {fmt_f(row["Split"])}) {{')
            recurse(yn,indent+1); lines.append('    '*indent+'} else {'); recurse(nn,indent+1); lines.append('    '*indent+'}')
        recurse(root,1); lines.append('}\n')
    lines.append('float predict_proba_one(const float* X) {')
    lines.append(f'    float score = {fmt_f(base_logit)};')
    for tid in tree_ids: lines.append(f'    score += tree_{tid}(X);')
    lines.append('    return 1.0f / (1.0f + expf(-score));'); lines.append('}')
    lines.append(_timing_func_c())
    return ForgeAOT(compile_c('\n'.join(lines), prefix='xgforge_'))


In [ ]:
"""Compile a head-aware **Northside** model (GP trees -> stacked LR head) to a fp32 .so,
reusing XGForge's compile/load plumbing. This is Northside's RL core inference path.

clamp_ops:
  False (default, fast)  -- emit each GP op as bare C arithmetic. The genetic engine clamps
                            every op to +/-1e6 during EVOLUTION to stop runaway candidates from
                            poisoning the population with inf/NaN. A trained, selected model's
                            ops stay well inside that band, so the clamp never fires at inference
                            and is pure overhead (~2x latency). Verified AUC-identical on test sets.
  True  (exact)          -- wrap every op in clampf(+/-1e6), bit-faithful to the numpy reference's
                            per-op clip. Use only if a model genuinely relies on op saturation.
The 'clamp' OP itself (semantic clamp to [0,1]) always clamps, in both modes.
"""
import numpy as np

def _node_c(node, nf, clamp_ops):
    if node.t == 'in':  return f"X[{node.v % nf}]"
    if node.t == 'cn':  return f"({fmt_f(node.v)})"
    a = [_node_c(c, nf, clamp_ops) for c in node.c]; op = node.v
    if   op == 'fma':   e = f"({a[0]}*{a[1]}+{a[2]})"
    elif op == 'add':   e = f"({a[0]}+{a[1]})"
    elif op == 'mul':   e = f"({a[0]}*{a[1]})"
    elif op == 'sub':   e = f"({a[0]}-{a[1]})"
    elif op == 'max':   e = f"fmaxf({a[0]},{a[1]})"
    elif op == 'min':   e = f"fminf({a[0]},{a[1]})"
    elif op == 'neg':   e = f"(-({a[0]}))"
    elif op == 'abs':   e = f"fabsf({a[0]})"
    elif op == 'clamp': return f"clamp01({a[0]})"                 # semantic [0,1]; never needs +/-1e6
    elif op == 'lt':    e = f"(({a[0]})<({a[1]})?1.0f:0.0f)"
    elif op == 'ifte':  e = f"(({a[0]})>0.0f?{a[1]}:{a[2]})"
    else:               e = a[0]
    return f"clampf({e})" if clamp_ops else e

def rl_to_c(h, nf, clamp_ops=False):
    sc, lr = h.rl_sc, h.rl_lr
    exprs = [_node_c(t, nf, clamp_ops) for m in h.members for t in m.trees]   # stack() order
    D = len(exprs)
    std = np.where(np.abs(sc.scale_) > 1e-12, sc.scale_, 1.0)
    wf = lr.coef_[0] / std                       # fold StandardScaler into LR weights
    bf = float(lr.intercept_[0]) - float(np.sum(wf * sc.mean_))
    L = ['#include <math.h>', '#include <stdint.h>', '#include <time.h>', '',
         'static inline float clamp01(float x){return x<0.0f?0.0f:(x>1.0f?1.0f:x);}']
    if clamp_ops:
        L.append('static inline float clampf(float x){return x<-1e6f?-1e6f:(x>1e6f?1e6f:x);}')
    L += ['', 'float predict_proba_one(const float* X) {']
    for i, e in enumerate(exprs): L.append(f'    float s{i} = {e};')
    logit = ' + '.join([fmt_f(bf)] + [f'{fmt_f(wf[i])}*s{i}' for i in range(D)])
    L += [f'    float logit = {logit};', '    return 1.0f/(1.0f+expf(-logit));', '}', _timing_func_c()]
    return ForgeAOT(compile_c('\n'.join(L), prefix='rl_'))


### 4 · Will Northside help *my* data?
We tested `gate_auc` as the predictor of the accuracy gain — it didn't generalize. **Headroom** (how far XGBoost is below 1.0) does. So estimate the gain from XGBoost's CV AUC, before you even fit Northside.

In [ ]:
# ── Northside advisory ─────────────────────────────────────────────────────────
# Calibrated on 19 OpenML binary datasets. We TESTED gate_auc (whether x can pick the
# better expert) as a predictor of Northside's gain — it did NOT generalize (corr −0.30,
# leave-one-out R² negative). What predicts the gain is HEADROOM = how far XGBoost sits
# below 1.0. (Part of that is real signal the GP members add; part is the unavoidable
# ceiling near AUC 1.0.) So you can estimate the gain BEFORE fitting Northside, from
# XGBoost's own cross-val AUC. Treat it as a rough guide (±~0.005), not a guarantee.
_ADV = (-0.0014, 0.0595)   # ΔAUC ≈ a + b·headroom

def northside_advisory(xgb_cv_auc: float):
    head = 1.0 - xgb_cv_auc
    est  = _ADV[0] + _ADV[1] * head
    if   xgb_cv_auc >= 0.97: note = "near the ceiling — expect a tie; take Northside for speed/size/energy, not accuracy."
    elif est >= 0.008:       note = f"good candidate — expect ≈ +{est:.3f} AUC over XGBoost (±~0.005)."
    elif est >= 0.003:       note = f"marginal — maybe ≈ +{est:.3f} AUC; worth a CV check. Speed win still applies."
    else:                    note = "little accuracy gain expected; Northside's value here is latency/size/energy."
    return est, note

for a in (0.70, 0.80, 0.86, 0.92, 0.97, 0.995):
    est, note = northside_advisory(a)
    print(f"  XGBoost CV AUC {a:.3f}  →  est ΔAUC {est:+.3f}   {note}")

### 5 · Use Northside on your own data
Load → (optional) advisory → fit → predict → compile to a ~15 KB `.so` that predicts in nanoseconds.

In [ ]:
# ── Use Northside on YOUR data ───────────────────────────────────────────────
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# 1) Bring data as X (float matrix) and y (binary 0/1). Here we borrow an OpenML set;
#    for your own:  X = df[feature_cols].to_numpy('float32');  y = df[target].to_numpy()
Xtr, ytr, Xte, yte = load_openml(37, seed=0)      # pima-diabetes (loader from §2: preprocessed + split)
# for your own data:  X = df[feature_cols].to_numpy('float32'); y = df[target].to_numpy()
#                     Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

# 2) Should you even expect an accuracy gain? Ask the advisory from XGBoost's AUC.
xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                    eval_metric='logloss', verbosity=0).fit(Xtr, ytr)
xgb_auc = roc_auc_score(yte, xgb.predict_proba(Xte)[:, 1])
print(f"XGBoost AUC {xgb_auc:.4f}  →  {northside_advisory(xgb_auc)[1]}")

# 3) Fit Northside (the gated RL + XGBoost hybrid). Bump members/pop/gens for more capacity.
ns = RoutedHybrid(gate='lr', rl_kw=dict(members=12, pop=150, gens=18), seed=0).fit(Xtr, ytr)

# 4) Predict. Three serving modes — soft blend is the accurate one (RLN); proba_xgb is XGBoost alone.
print(f"  soft-MoE (RLN) AUC : {roc_auc_score(yte, ns.proba_soft(Xte)):.4f}")
print(f"  XGBoost-only  AUC : {roc_auc_score(yte, ns.proba_xgb(Xte)):.4f}")

# 5) Ship the nanosecond core: compile Northside's RL expert to a ~15 KB, zero-dep .so.
rl_so = rl_to_c(ns, Xtr.shape[1])                 # fast inference default (clamp_ops=False)
print(f"  compiled Northside: {rl_so.so_kb:.1f} KB, {rl_so.time_ns(Xte[0].astype('float32')):.0f} ns/prediction (pure C)")
#   rl_so.predict_proba_one(x) -> prob ;  for the heavy path compile XGBoost too:  xgforge(ns.xgb)  (=RLNF)

### 6 · Rigorous check — repeated CV + paired significance (optional)
Pre-registered single policy (soft-MoE), paired bootstrap + DeLong. `evaluate_cv(X, y, ...)` returns ΔAUC with CIs and a win/loss/tie verdict.

In [ ]:
#!/usr/bin/env python3
"""Turn 'looks better' into 'is better': repeated stratified k-fold CV with a
PRE-SPECIFIED policy (soft-MoE), scored by a paired test (bootstrap + DeLong) on
the RLN-minus-XGBoost AUC difference. Imports the hybrid for local testing; the
notebook inlines everything."""
import numpy as np
from math import erf, sqrt
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml, load_breast_cancer

set_ops(False)

# ---------- full-dataset loaders (CV does its own splitting) ----------
def _prep_full(Xdf, y):
    cat = Xdf.select_dtypes(include=['object', 'category']).columns.tolist()
    num = Xdf.select_dtypes(include=['number']).columns.tolist()
    for c in cat: Xdf[c] = Xdf[c].fillna(Xdf[c].mode().iloc[0])
    for c in num: Xdf[c] = Xdf[c].fillna(Xdf[c].median())
    pre = ColumnTransformer([('n', StandardScaler(), num),
                             ('c', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat)])
    X = pre.fit_transform(Xdf).astype('float32')   # unsupervised prep on full data (no label leakage)
    return X, np.asarray(y).astype(int)

def load_full_openml(data_id, sub=None, seed=0):
    d = fetch_openml(data_id=data_id, as_frame=True, parser='auto')
    y = d.target.astype('category').cat.codes.values
    X, y = _prep_full(d.data.copy(), y)
    if sub and sub < len(X):
        from sklearn.model_selection import train_test_split
        X, _, y, _ = train_test_split(X, y, train_size=sub, random_state=seed, stratify=y)
    return X, y

def load_full_breast():
    import pandas as pd
    bc = load_breast_cancer()
    return _prep_full(pd.DataFrame(bc.data, columns=bc.feature_names), bc.target)

# ---------- paired significance tests on AUC difference ----------
def paired_bootstrap_auc(y, p_a, p_b, B=1000, seed=0):
    """Stratified paired bootstrap of AUC(a)-AUC(b). Returns (diff, (lo,hi), p_one_sided)."""
    rng = np.random.default_rng(seed)
    pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    diff0 = roc_auc_score(y, p_a) - roc_auc_score(y, p_b)
    d = np.empty(B)
    for b in range(B):
        idx = np.concatenate([rng.choice(pos, len(pos), True), rng.choice(neg, len(neg), True)])
        d[b] = roc_auc_score(y[idx], p_a[idx]) - roc_auc_score(y[idx], p_b[idx])
    lo, hi = np.percentile(d, [2.5, 97.5])
    return diff0, (lo, hi), float(np.mean(d <= 0))   # one-sided: evidence against a>b

def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1; i = j
    out = np.empty(N); out[J] = T; return out

def delong_test(y, p_a, p_b):
    """Fast DeLong for two correlated AUCs. Returns (diff, (lo,hi), p_two_sided, (auc_a,auc_b))."""
    y = np.asarray(y); order = (-y).argsort(kind='mergesort')   # positives first
    P = np.vstack([p_a, p_b])[:, order]; m = int(y.sum()); n = len(y) - m; k = 2
    pos, neg = P[:, :m], P[:, m:]
    tx = np.vstack([_midrank(pos[r]) for r in range(k)])
    ty = np.vstack([_midrank(neg[r]) for r in range(k)])
    tz = np.vstack([_midrank(P[r])   for r in range(k)])
    aucs = (tz[:, :m].sum(1) - m * (m + 1) / 2.0) / (m * n)
    v01 = (tz[:, :m] - tx) / n           # (k, m)
    v10 = 1.0 - (tz[:, m:] - ty) / m     # (k, n)
    cov = np.cov(v01) / m + np.cov(v10) / n
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    diff = aucs[0] - aucs[1]; se = sqrt(var) if var > 0 else 0.0
    z = diff / se if se > 0 else 0.0
    p = 2 * (1 - 0.5 * (1 + erf(abs(z) / sqrt(2))))
    return diff, (diff - 1.96 * se, diff + 1.96 * se), p, (aucs[0], aucs[1])

# ---------- the CV harness ----------
def evaluate_cv(X, y, n_splits=5, repeats=3, gate='lr', rl_kw=None, policy='soft', seed0=0, B=1000, name=""):
    """Repeated stratified k-fold OOF for XGB vs a PRE-SPECIFIED RLN policy, then
    a paired bootstrap + DeLong on the AUC difference."""
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=repeats, random_state=seed0)
    N = len(y)
    oof_x = [np.full(N, np.nan) for _ in range(repeats)]
    oof_r = [np.full(N, np.nan) for _ in range(repeats)]
    for fi, (tr, te) in enumerate(rskf.split(X, y)):
        rep = fi // n_splits
        h = RoutedHybrid(gate=gate, rl_kw=rl_kw, seed=seed0 + fi).fit(X[tr], y[tr])
        oof_x[rep][te] = h.proba_xgb(X[te])
        oof_r[rep][te] = h.proba_soft(X[te]) if policy == 'soft' else h.proba_stack(X[te])
    auc_x = [roc_auc_score(y, oof_x[r]) for r in range(repeats)]
    auc_r = [roc_auc_score(y, oof_r[r]) for r in range(repeats)]
    p_x = np.nanmean(np.vstack(oof_x), 0); p_r = np.nanmean(np.vstack(oof_r), 0)  # repeat-averaged OOF
    bdiff, bci, bp = paired_bootstrap_auc(y, p_r, p_x, B=B, seed=seed0)
    ddiff, dci, dp, (ar, ax) = delong_test(y, p_r, p_x)
    print(f"\n=== {name} | {repeats}x{n_splits}-fold CV | policy='{policy}' (pre-specified) ===")
    print(f"  XGBoost AUC (OOF): {np.mean(auc_x):.4f} ± {np.std(auc_x):.4f}  across repeats")
    print(f"  RLN     AUC (OOF): {np.mean(auc_r):.4f} ± {np.std(auc_r):.4f}")
    print(f"  ΔAUC (RLN − XGB):  {bdiff:+.4f}")
    print(f"    paired bootstrap 95% CI [{bci[0]:+.4f}, {bci[1]:+.4f}]   one-sided p={bp:.4f}")
    print(f"    DeLong           95% CI [{dci[0]:+.4f}, {dci[1]:+.4f}]   two-sided p={dp:.4f}")
    sig = bci[0] > 0 and dci[0] > 0
    print(f"  verdict: {'REAL — CI clears 0 (both tests)' if sig else 'not significant — CI includes 0'}")
    return dict(name=name, auc_x=np.mean(auc_x), auc_r=np.mean(auc_r), diff=bdiff,
                boot_ci=bci, boot_p=bp, delong_ci=dci, delong_p=dp, sig=sig)


### 7 · Speed vs the world
Same 50-tree model where it's an apples-to-apples AOT shootout. `py-call µs` includes ~Python/ctypes overhead (the same floor for every compiled model); `pure-C ns` is the real in-loop deployment latency. Timber's number is its published figure on this exact config — for a strict head-to-head run `timber bench` on the identical model.

In [ ]:
# ── The stopwatch: latency / throughput / energy vs the leaders ───────────────
import time, os, tempfile, warnings, numpy as np
warnings.filterwarnings('ignore')
import xgboost as xgb, lightgbm as lgb
from sklearn.metrics import roc_auc_score
def _bench(fn, Xte, w=300, n=4000):
    for i in range(w): fn(Xte[i%len(Xte):i%len(Xte)+1])
    t=time.perf_counter()
    for i in range(n): fn(Xte[i%len(Xte):i%len(Xte)+1])
    return (time.perf_counter()-t)/n*1e6
def latency_shootout(Xtr, ytr, Xte, yte, n_trees=50, rl_kw=None):
    rows={}
    clf=xgb.XGBClassifier(n_estimators=n_trees,max_depth=4,learning_rate=0.1,eval_metric='logloss',
                          random_state=0,verbosity=0).fit(Xtr,ytr); bst=clf.get_booster()
    rows['XGBoost native (py)']=dict(us=_bench(lambda r: bst.inplace_predict(r),Xte),ns=None,
                                     auc=roc_auc_score(yte,clf.predict_proba(Xte)[:,1]),kb=None)
    xf=xgforge(clf)
    rows['XGForge .so (ours)']=dict(us=_bench(lambda r: xf.predict_proba_one(r[0]),Xte),ns=xf.time_ns(Xte[0]),
                                    auc=roc_auc_score(yte,xf.predict_proba(Xte)),kb=xf.so_kb)
    try:
        import treelite, tl2cgen
        tlm=treelite.frontend.from_xgboost(bst); lib=os.path.join(tempfile.mkdtemp(),'tl.so')
        tl2cgen.export_lib(tlm,toolchain='gcc',libpath=lib,params={'parallel_comp':1},verbose=False)
        pr=tl2cgen.Predictor(lib)
        def tlc(r): return pr.predict(tl2cgen.DMatrix(r))
        auc=roc_auc_score(yte,np.array([float(np.ravel(tlc(Xte[i:i+1]))[0]) for i in range(len(Xte))]))
        rows['treelite/tl2cgen .so']=dict(us=_bench(tlc,Xte),ns=None,auc=auc,kb=os.path.getsize(lib)/1024)
    except Exception as e: print("tl2cgen skipped:",type(e).__name__)
    lgm=lgb.LGBMClassifier(n_estimators=n_trees,max_depth=4,learning_rate=0.1,verbosity=-1,random_state=0).fit(Xtr,ytr)
    lbst=lgm.booster_
    rows['LightGBM native (py)']=dict(us=_bench(lambda r: lbst.predict(r),Xte),ns=None,
                                      auc=roc_auc_score(yte,lgm.predict_proba(Xte)[:,1]),kb=None)
    h=RoutedHybrid(gate='lr',rl_kw=rl_kw or dict(members=10,pop=120,gens=14),seed=0).fit(Xtr,ytr)
    rl=rl_to_c(h, Xtr.shape[1])
    rows['Northside .so']=dict(us=_bench(lambda r: rl.predict_proba_one(r[0]),Xte),ns=rl.time_ns(Xte[0]),
                               auc=roc_auc_score(yte,rl.predict_proba(Xte)),kb=rl.so_kb)
    base=rows['XGBoost native (py)']['us']
    print(f"\n{'engine':24s} {'py-call µs':>11s} {'pure-C ns':>10s} {'thru/s(py)':>11s} {'vs XGB':>7s} {'AUC':>7s} {'KB':>6s}")
    for k,v in rows.items():
        print(f"{k:24s} {v['us']:>11.3f} {(('%.0f'%v['ns']) if v['ns'] else '—'):>10s} {1e6/v['us']:>11,.0f} "
              f"{base/v['us']:>6.0f}x {v['auc']:>7.4f} {(('%.1f'%v['kb']) if v['kb'] else '—'):>6s}")
    print("Timber (published, same 50-tree breast-cancer config): ~2.0 µs/call, ~48 KB  [kossisoroyce/timber]")
    for k,v in rows.items():
        if v['ns']: print(f"  {k}: pure-C ≈ {1e9/v['ns']:,.0f} preds/s/core; ≈ {base*1000/v['ns']:,.0f}× less active time (energy) than native XGBoost")
    return rows

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
bc=load_breast_cancer(); Xbc=StandardScaler().fit_transform(bc.data).astype('float32'); ybc=bc.target
Xtr,Xte,ytr,yte=train_test_split(Xbc,ybc,test_size=0.4,random_state=0,stratify=ybc)
rows=latency_shootout(Xtr,ytr,Xte,yte,n_trees=50)
import matplotlib.pyplot as plt
names=list(rows); us=[rows[k]['us'] for k in names]
c=['#d4622a' if 'Northside' in k else ('#e0a07a' if 'ours' in k else '#9a9a9a') for k in names]
fig,ax=plt.subplots(figsize=(9,4)); ax.barh(names,us,color=c); ax.set_xscale('log'); ax.invert_yaxis()
ax.set_xlabel('single-sample latency, µs (log) — lower is better'); ax.axvline(2.0,ls='--',c='#3a6ea5',lw=1)
ax.text(2.0,len(names)-0.4,' Timber pub.',color='#3a6ea5',fontsize=8)
ax.set_title('Inference latency: Northside & XGForge (ours) vs native leaders'); plt.tight_layout(); plt.show()

### 8 · What we found — honestly

**Speed is the unconditional win.** Compiled, Northside is the fastest and smallest predictor here — tens of nanoseconds, ~10–35 M predictions/sec/core, ~15 KB, zero deps, microcontroller-capable — orders of magnitude under any native-Python predictor. Energy tracks active time, so the per-prediction energy advantage tracks the latency.

**Accuracy is conditional — and predictable.** Across 19 OpenML binary datasets, RLN significantly beats XGBoost on **6**, loses on **1** (phoneme, by −0.0024 — negligible), and ties on **12** (both tests: paired bootstrap + DeLong). The wins (+0.015 to +0.023 on the hard sets) cluster where XGBoost has **headroom**; where it's near-ceiling, it's a tie. We *tested* whether the gate's separability (`gate_auc`) predicts the gain — it does **not** (corr −0.30); headroom does (corr +0.76). Part of that is genuine signal the GP members add, part is the unavoidable ceiling near AUC 1.0.

In [ ]:
# ── What the full 19-dataset study found (your run, embedded so this reproduces) ──
import numpy as np, matplotlib.pyplot as plt
# name, ΔAUC, boot_lo, boot_hi, delong_p
B=[('pima-diabetes',0.0150,0.0074,0.0219,0.0000),('german-credit',0.0060,-0.0036,0.0152,0.2065),
('wdbc',0.0007,-0.0003,0.0018,0.1726),('banknote',0.0002,0.0000,0.0006,0.1960),
('blood-transf',0.0196,0.0086,0.0317,0.0009),('ilpd',0.0099,-0.0038,0.0239,0.1699),
('tic-tac-toe',-0.0008,-0.0022,-0.0000,0.1280),('kr-vs-kp',-0.0000,-0.0001,0.0001,0.7399),
('qsar-biodeg',0.0025,-0.0009,0.0061,0.1634),('pc1',-0.0069,-0.0177,0.0055,0.2556),
('kc1',0.0227,0.0136,0.0330,0.0000),('pc3',0.0161,0.0043,0.0286,0.0121),
('pc4',0.0042,-0.0002,0.0092,0.0600),('phoneme',-0.0024,-0.0032,-0.0015,0.0000),
('churn',0.0003,-0.0034,0.0046,0.8881),('ozone-8hr',0.0072,-0.0015,0.0178,0.1291),
('spambase',-0.0003,-0.0009,0.0005,0.4787),('adult',0.0043,0.0027,0.0061,0.0000),
('bank-marketing',0.0048,0.0022,0.0077,0.0009)]
# name, xgb_auc, gate_auc, ΔAUC  (gate_auc measured at fit time)
P=[('pima',0.793,0.686,0.0150),('german',0.770,0.521,0.0060),('wdbc',0.991,0.672,0.0007),
('banknote',1.000,0.766,0.0002),('blood-tr',0.683,0.576,0.0196),('ilpd',0.712,0.648,0.0099),
('tic-tac',0.998,0.685,-0.0008),('kr-vs-kp',0.999,0.905,-0.0000),('qsar',0.922,0.558,0.0025),
('pc1',0.858,0.687,-0.0069),('kc1',0.765,0.595,0.0227),('pc3',0.809,0.715,0.0161),
('pc4',0.936,0.789,0.0042),('phoneme',0.950,0.650,-0.0024),('churn',0.913,0.633,0.0003),
('ozone',0.904,0.697,0.0072),('spambase',0.987,0.669,-0.0003),('adult',0.906,0.707,0.0043),
('bank',0.912,0.742,0.0048)]
def verdict(d,lo,hi,p): return 'win' if (lo>0 and p<0.05 and d>0) else ('loss' if (hi<0 and p<0.05 and d<0) else 'ns')
COL={'win':'#d4622a','loss':'#3a6ea5','ns':'#9a9a9a'}
fig,(a1,a2)=plt.subplots(1,2,figsize=(14,7),gridspec_kw={'width_ratios':[1,1.05]})
nw=nl=0
for i,(nm,d,lo,hi,p) in enumerate(B):
    v=verdict(d,lo,hi,p); nw+=v=='win'; nl+=v=='loss'
    a1.plot([lo,hi],[i,i],color=COL[v],lw=2.3); a1.plot(d,i,'o',color=COL[v],ms=5.5)
a1.axvline(0,color='grey',ls='--',lw=1); a1.set_yticks(range(len(B))); a1.set_yticklabels([b[0] for b in B],fontsize=8); a1.invert_yaxis()
a1.set_xlabel('ΔAUC (RLN − XGBoost), 95% CI'); a1.set_title(f'Where Northside beats XGBoost  ({nw} wins · {nl} loss · {len(B)-nw-nl} ties)')
xgb=np.array([r[1] for r in P]); d=np.array([r[3] for r in P]); head=1-xgb; ga=np.array([r[2] for r in P])
cc=['#d4622a' if x>0.004 else ('#3a6ea5' if x<-0.001 else '#9a9a9a') for x in d]
a2.scatter(head,d*1000,c=cc,s=60,zorder=3); a2.axhline(0,c='grey',lw=1)
a2.plot(np.sort(head),np.polyval(np.polyfit(head,d*1000,1),np.sort(head)),'--',c='#d4622a',lw=1.5)
a2.set_xlabel('headroom = 1 − XGBoost AUC'); a2.set_ylabel('ΔAUC ×1000')
a2.set_title(f'Headroom predicts the gain (r={np.corrcoef(head,d)[0,1]:+.2f}); gate_auc did not (r={np.corrcoef(ga,d)[0,1]:+.2f})')
for nm,xa,g,dd in P: a2.annotate(nm,(1-xa,dd*1000),fontsize=6.6,xytext=(3,2),textcoords='offset points')
plt.tight_layout(); plt.show()

### 9 · License

Released under the **GNU AGPL-3.0**. You may use, study, modify, and redistribute it freely; if you modify it and distribute it
**or offer it over a network/as a service**, you must release your source under the same license. This keeps improvements open and
prevents anyone from taking Northside proprietary and walking away with it.

**Commercial licensing:** if you want to use Northside in a closed-source or proprietary product without the AGPL's copyleft
obligations, a separate commercial license is available — contact **[your name / email]**. (Dual licensing is possible because the
authors hold the copyright; external contributions are accepted under a CLA so this remains possible.)

```
Copyright (C) 2026 [your name]
This program is free software: you can redistribute it and/or modify it under the terms of the
GNU Affero General Public License as published by the Free Software Foundation, version 3.
This program is distributed WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See <https://www.gnu.org/licenses/>.
```
Dependencies (XGBoost, LightGBM, scikit-learn, NumPy, treelite/TL2cgen) are permissively licensed and impose no constraint on this choice. Timber is referenced for benchmarking, not bundled.